# 00 - Data acquisition

**Goal:** download the raw files into `data/raw` and record what each file contains.

**Source:** Kaggle dataset *Air Pollution in Seoul* (`bappekim/air-pollution-in-seoul`), <https://www.kaggle.com/datasets/bappekim/air-pollution-in-seoul>. According to its description, the data comes from the Seoul Metropolitan Government. The terms of use are listed on the Kaggle page and are not restated here.

The raw files are not committed to the repository; this notebook downloads them again when they are missing.

## 1. Imports

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from seoul_air import config
from seoul_air.data import download_raw, load_raw
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 140)
config.FIGURES.mkdir(parents=True, exist_ok=True)
config.METRICS.mkdir(parents=True, exist_ok=True)

## 2. Download

In [2]:
paths = download_raw()
for p in paths:
    print(f"{p.name:32s} {p.stat().st_size / 1e6:8.1f} MB")

Raw files already present, download skipped
Measurement_info.csv                124.4 MB
Measurement_item_info.csv             0.0 MB
Measurement_station_info.csv          0.0 MB
Measurement_summary.csv              92.7 MB


## 3. File inventory

In [3]:
inventory = []
for key, name in config.RAW_FILES.items():
    df = load_raw(name)
    inventory.append({"file": name, "rows": len(df), "columns": df.shape[1],
                      "column_names": ", ".join(df.columns)})
pd.DataFrame(inventory)

,file,rows,columns,column_names
0,Measurement_info.csv,3885066,5,"Measurement date, Station code, Item code, Ave..."
1,Measurement_item_info.csv,6,7,"Item code, Item name, Unit of measurement, Goo..."
2,Measurement_station_info.csv,25,5,"Station code, Station name(district), Address,..."
3,Measurement_summary.csv,647511,11,"Measurement date, Station code, Address, Latit..."


**Interpretation.** `Measurement_info.csv` (3,885,066 rows) is the original long table: one row per station, hour and pollutant, with an instrument status code. `Measurement_summary.csv` (647,511 rows) is a wide version of the same readings without the status. The two small files describe the 6 pollutants (with their grade thresholds) and the 25 stations. The analysis uses the original long table, for the reason shown in notebook 01.